[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/10_decision_trees/03_regression_trees.ipynb)

# Bài 03 — Regression Tree

**🎯 Mục tiêu**
- Mở rộng cây cho hồi quy: split theo **giảm phương sai** (MSE), lá dự đoán **trung bình**.
- Thấy dự đoán dạng **bậc thang** (hằng số từng khúc) và ảnh hưởng của độ sâu.
- Cài đặt split hồi quy from scratch bằng tổng tích luỹ; so sánh criterion MSE vs MAE (robust với outlier).
- Hiểu vì sao regression tree **không ngoại suy** — và hệ quả cho dữ liệu chuỗi thời gian.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/10_decision_trees

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor, plot_tree

from ml_course.datasets import make_regression_1d
from ml_course.visualization import set_style

set_style()

## 1. Từ phân loại sang hồi quy

| | Classification tree | Regression tree |
|---|---|---|
| Impurity của node | Gini / entropy | **phương sai** $\frac1n\sum(y_i - \bar y)^2$ (MSE) |
| Dự đoán ở lá | lớp đa số / tỉ lệ lớp | **trung bình** $\bar y$ (MSE) hoặc **trung vị** (MAE) |

Gain của split = giảm phương sai: $\text{Var}(P) - \left[\frac{n_L}{n}\text{Var}(L) + \frac{n_R}{n}\text{Var}(R)\right]$ — tương đương chọn split làm **tổng bình phương sai số** (SSE) nhỏ nhất khi mỗi bên dự đoán bằng trung bình của nó.

In [ ]:
X, y = make_regression_1d(n=150, kind="sin", noise=0.3, seed=0)
grid = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
for ax, depth in zip(axes, [1, 2, 4, None]):
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, y)
    ax.scatter(X, y, s=10, alpha=0.6)
    ax.plot(grid, tree.predict(grid), "r", lw=2)
    ax.set_title(f"max_depth = {depth} | {tree.get_n_leaves()} lá")
plt.show()

Độ sâu $d$ → tối đa $2^d$ lá → hàm bậc thang tối đa $2^d$ bậc. Không giới hạn → mỗi điểm một bậc → bám theo nhiễu.

## 2. Split hồi quy from scratch

Với dữ liệu đã sắp xếp theo $x$, tổng và tổng bình phương của $y$ bên trái tại mọi vị trí cắt là `cumsum` → phương sai hai bên cho mọi ngưỡng trong $O(n)$:

$$\operatorname{Var} = \frac{\sum y^2}{n} - \left(\frac{\sum y}{n}\right)^2$$

In [ ]:
def best_regression_split(x, y):
    order = np.argsort(x, kind="mergesort")
    xs, ys = x[order], y[order]
    n = len(y)
    n_left = np.arange(1, n)
    s_left, sq_left = np.cumsum(ys)[:-1], np.cumsum(ys**2)[:-1]
    s_right, sq_right = ys.sum() - s_left, (ys**2).sum() - sq_left
    var_left = sq_left / n_left - (s_left / n_left) ** 2
    var_right = sq_right / (n - n_left) - (s_right / (n - n_left)) ** 2
    child = (n_left * var_left + (n - n_left) * var_right) / n
    child[xs[1:] == xs[:-1]] = np.inf
    i = int(np.argmin(child))
    return (xs[i] + xs[i + 1]) / 2, ys.var() - child[i]


t, gain = best_regression_split(X.ravel(), y)
stump = DecisionTreeRegressor(max_depth=1).fit(X, y)
print(f"scratch: ngưỡng = {t:.4f}, giảm phương sai = {gain:.4f}")
print(f"sklearn: ngưỡng = {stump.tree_.threshold[0]:.4f}")

## 3. Đọc một regression tree: giá nhà

In [ ]:
houses = pd.read_csv("../../data/sample/house_prices.csv")
features = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
reg = DecisionTreeRegressor(max_depth=3, min_samples_leaf=20, random_state=0).fit(houses[features], houses["price_billion_vnd"])
fig, ax = plt.subplots(figsize=(20, 7))
plot_tree(reg, feature_names=features, filled=True, fontsize=8, precision=2, ax=ax)
plt.show()

Mỗi lá là một "phân khúc" nhà với giá trung bình riêng — dễ giải thích cho người không chuyên. Nhưng chỉ có $2^3 = 8$ mức giá khác nhau!

## 4. Criterion: MSE vs MAE

`criterion="squared_error"` (mặc định) dùng trung bình — nhạy với outlier; `criterion="absolute_error"` dùng **trung vị** ở lá — robust hơn nhưng train chậm hơn nhiều.

In [ ]:
rng = np.random.default_rng(3)
Xo, yo = make_regression_1d(n=200, kind="sin", noise=0.2, seed=3)
outliers = rng.choice(200, 12, replace=False)
yo_dirty = yo.copy()
yo_dirty[outliers] += 8                                   # 6% điểm bị lỗi nhập liệu (+8)
grid = np.linspace(Xo.min(), Xo.max(), 500).reshape(-1, 1)
plt.scatter(Xo, yo_dirty, s=10, alpha=0.6)
for crit, color in [("squared_error", "tab:red"), ("absolute_error", "tab:green")]:
    m = DecisionTreeRegressor(criterion=crit, max_depth=4, random_state=0).fit(Xo, yo_dirty)
    err = np.mean(np.abs(m.predict(Xo) - yo))
    plt.plot(grid, m.predict(grid), color=color, lw=2, label=f"{crit} — MAE so với y sạch = {err:.3f}")
plt.ylim(-4.5, 12)
plt.legend()
plt.title("Outlier kéo lá MSE lên; lá MAE (trung vị) gần như không bị ảnh hưởng")
plt.show()

## 5. Không ngoại suy

Lá dự đoán trung bình các $y$ đã thấy → ngoài vùng dữ liệu train, cây trả về **hằng số** của lá ngoài cùng. Với dữ liệu có xu hướng (doanh số tăng dần theo thời gian), đây là thất bại nghiêm trọng:

In [ ]:
t_train = np.arange(0, 100).reshape(-1, 1)
sales = 50 + 2.0 * t_train.ravel() + rng.normal(0, 8, 100)
t_future = np.arange(100, 140).reshape(-1, 1)
tree = DecisionTreeRegressor(max_depth=5).fit(t_train, sales)
lin = LinearRegression().fit(t_train, sales)
plt.scatter(t_train, sales, s=10, label="train")
plt.plot(np.r_[t_train, t_future], tree.predict(np.r_[t_train, t_future]), "r", label="Regression tree")
plt.plot(np.r_[t_train, t_future], lin.predict(np.r_[t_train, t_future]), "g--", label="Linear regression")
plt.axvline(99.5, color="gray", ls=":")
plt.xlabel("thời gian")
plt.legend()
plt.title("Tương lai: cây dự đoán đường phẳng")
plt.show()
print(f"t = 139: cây dự đoán {tree.predict([[139]])[0]:.1f} | xu hướng thật ≈ {50 + 2 * 139}")

**Khắc phục:** dự đoán **mức thay đổi** (sai phân) thay vì mức tuyệt đối, khử xu hướng trước rồi cho cây học phần dư, hoặc kết hợp với model tuyến tính (chương 25 — time series). Điều này áp dụng cho **mọi** model dựa trên cây (Random Forest, Gradient Boosting).

## 6. Regression tree vs Linear Regression

In [ ]:
for name, model in [("Linear Regression", LinearRegression()),
                    ("Tree depth 3", DecisionTreeRegressor(max_depth=3, random_state=0)),
                    ("Tree depth 6", DecisionTreeRegressor(max_depth=6, random_state=0)),
                    ("Tree không giới hạn", DecisionTreeRegressor(random_state=0))]:
    mae = -cross_val_score(model, houses[features], houses["price_billion_vnd"], cv=5, scoring="neg_mean_absolute_error").mean()
    print(f"{name:20s} MAE (5-fold CV) = {mae:.3f} tỉ")

Cây độ sâu vừa phải thắng Linear Regression (giá nhà có quan hệ phi tuyến và tương tác — vd diện tích × khoảng cách tới trung tâm — mà cây tự bắt được); cây quá nông underfit, cây không giới hạn overfit. Độ phức tạp của cây phải được chọn bằng CV (bài 04).

## 🧠 Kiểm tra nhanh

1. Lá của regression tree (criterion MSE) dự đoán giá trị gì? Vì sao giá trị đó tối ưu cho MSE?
2. Một cây độ sâu 4 có tối đa bao nhiêu giá trị dự đoán khác nhau?
3. Cây được train trên giá nhà 2–20 tỉ. Dự đoán cho một căn biệt thự có thể vượt 20 tỉ không?
4. Khi nào nên dùng `criterion="absolute_error"`?

## 📌 Tóm tắt

- Regression tree: split giảm phương sai; lá = trung bình (MSE) hoặc trung vị (MAE).
- Dự đoán hằng số từng khúc; độ sâu điều khiển số bậc.
- Không ngoại suy — cẩn thận với dữ liệu có xu hướng.